In [0]:
from pyspark.sql.functions import (
       col, trim, initcap, upper, when, to_timestamp, datediff,
       coalesce, lit, avg, first, current_timestamp
   )
   
print("Starting Silver layer transformations...")

In [0]:
 # 1. CUSTOMERS
print("\n--- silver.customers ---")
df = spark.table("bronze.customers")
df_silver = df \
       .withColumn("customer_city", initcap(trim(col("customer_city")))) \
       .withColumn("customer_state", upper(trim(col("customer_state")))) \
       .withColumn("customer_zip_code_prefix",
                   col("customer_zip_code_prefix").cast("string")) \
       .dropDuplicates(["customer_id"])
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.customers")
print(f"  Rows: {df_silver.count():,}")

In [0]:
# 2. ORDERS
print("\n--- silver.orders ---")

df = spark.table("bronze.orders")

df_silver = df \
    .withColumn("order_purchase_timestamp",
                to_timestamp(col("order_purchase_timestamp"))) \
    .withColumn("order_approved_at",
                to_timestamp(col("order_approved_at"))) \
    .withColumn("order_delivered_carrier_date",
                to_timestamp(col("order_delivered_carrier_date"))) \
    .withColumn("order_delivered_customer_date",
                to_timestamp(col("order_delivered_customer_date"))) \
    .withColumn("order_estimated_delivery_date",
                to_timestamp(col("order_estimated_delivery_date"))) \
    .withColumn("is_late_delivery",
                when(col("order_delivered_customer_date") >
                     col("order_estimated_delivery_date"), 1).otherwise(0)) \
    .withColumn("delivery_days",
                datediff(col("order_delivered_customer_date"),
                         col("order_purchase_timestamp"))) \
    .dropDuplicates(["order_id"])

df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.orders")

print(f"  Rows: {df_silver.count():,}")

In [0]:
# 3. ORDER ITEMS 
print("\n--- silver.order_items ---")

df = spark.table("bronze.order_items")

df_silver = (
    df
    .withColumn("price", col("price").cast("double"))
    .withColumn("freight_value", col("freight_value").cast("double"))
    .withColumn("total_item_value", col("price") + col("freight_value"))
    .dropDuplicates(["order_id", "order_item_id"])
)

df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.order_items")

print(f"  Rows: {df_silver.count():,}")

In [0]:
#  4. ORDER PAYMENTS
print("\n--- silver.order_payments ---")

df = spark.table("bronze.order_payments")

df_silver = df \
    .withColumn("payment_type",
                when(col("payment_type") == "not_defined", "unknown")
                .otherwise(col("payment_type"))) \
    .withColumn("payment_value", col("payment_value").cast("double")) \
    .withColumn("payment_installments",
                col("payment_installments").cast("int"))

df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.order_payments")

print(f"  Rows: {df_silver.count():,}")

In [0]:
# --- 5. ORDER REVIEWS ---
print("\n--- silver.order_reviews ---")

df = spark.table("bronze.order_reviews")

df_silver = (
    df
    # Convert score safely
    .withColumn("review_score", expr("try_cast(review_score as int)"))

    # Clean text fields (replace nulls)
    .withColumn("review_comment_title", coalesce(col("review_comment_title"), lit("")))
    .withColumn("review_comment_message", coalesce(col("review_comment_message"), lit("")))

    # Safe timestamp parsing (handles bad text values)
    .withColumn("review_creation_date", expr("try_cast(review_creation_date as timestamp)"))
    .withColumn("review_answer_timestamp", expr("try_cast(review_answer_timestamp as timestamp)"))

    # Optional: remove completely broken rows (if needed)
    # .filter(col("review_creation_date").isNotNull())

    .dropDuplicates(["review_id"])
)

# Save to Silver
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.order_reviews")

# Print count for sanity check
print(f"Rows: {df_silver.count():,}")

# --- 6. PRODUCTS ---
print("\n--- silver.products ---")

df_products = spark.table("bronze.products")

# Drop duplicate metadata columns from translation before join
df_translation = (
    spark.table("bronze.category_translation")
    .drop("batch_id", "ingestion_timestamp", "source_file")
)

df_silver = (
    df_products
    .join(df_translation, "product_category_name", "left")

    # Fix column name typos
    .withColumnRenamed("product_name_lenght", "product_name_length")
    .withColumnRenamed("product_description_lenght", "product_description_length")

    # Fill missing category names
    .withColumn(
        "product_category_name_english",
        coalesce(col("product_category_name_english"), lit("unknown"))
    )

    .dropDuplicates(["product_id"])
)

df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.products")

print(f"Rows: {df_silver.count():,}")


# --- 7. SELLERS ---
print("\n--- silver.sellers ---")

df = spark.table("bronze.sellers")

df_silver = (
    df
    .withColumn("seller_city", initcap(trim(col("seller_city"))))
    .withColumn("seller_state", upper(trim(col("seller_state"))))
    .withColumn(
        "seller_zip_code_prefix",
        col("seller_zip_code_prefix").cast("string")
    )
    .dropDuplicates(["seller_id"])
)

df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.sellers")
print(f"Rows: {df_silver.count():,}")


# --- 8. GEOLOCATION ---
print("\n--- silver.geolocation ---")

df = spark.table("bronze.geolocation")

df_silver = (
    df
    # Reduce huge dataset (~1M rows) → group by ZIP
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        avg("geolocation_lat").alias("latitude"),
        avg("geolocation_lng").alias("longitude"),
        first("geolocation_city").alias("city"),
        first("geolocation_state").alias("state")
    )
    .withColumn(
        "geolocation_zip_code_prefix",
        col("geolocation_zip_code_prefix").cast("string")
    )
)

df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.geolocation")
print(f"Rows: {df_silver.count():,} (reduced from ~1M)")


# --- 9. CATEGORY TRANSLATION ---
print("\n--- silver.category_translation ---")

df = spark.table("bronze.category_translation")

df.write.format("delta").mode("overwrite").saveAsTable("silver.category_translation")
print(f"Rows: {df.count():,}")


print("\n" + "="*60)
print("Silver layer transformations COMPLETE!")
print("="*60)

In [0]:
# 5. ORDER REVIEWS 
print("\n--- silver.order_reviews ---")

df = spark.table("bronze.order_reviews")
df.printSchema()

  

In [0]:
#  5. ORDER REVIEWS 
print("\n--- silver.order_reviews ---")

from pyspark.sql.functions import expr, coalesce, col, lit
df = spark.table("bronze.order_reviews")


df_silver = (
    df
    # Convert score safely
    .withColumn("review_score", expr("try_cast(review_score as int)"))

    # Clean text fields (replace nulls)
    .withColumn("review_comment_title", coalesce(col("review_comment_title"), lit("")))
    .withColumn("review_comment_message", coalesce(col("review_comment_message"), lit("")))

    # Safe timestamp parsing (handles bad text values)
    .withColumn("review_creation_date", expr("try_cast(review_creation_date as timestamp)"))
    .withColumn("review_answer_timestamp", expr("try_cast(review_answer_timestamp as timestamp)"))
     .dropDuplicates(["review_id"])
)

# Save to Silver
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.order_reviews")

# Print count for sanity check
print(f"Rows: {df_silver.count():,}")

In [0]:
#  6. PRODUCTS 
print("\n--- silver.products ---")

df_products = spark.table("bronze.products")

# Drop duplicate metadata columns from translation before join
df_translation = (
    spark.table("bronze.category_translation")
    .drop("batch_id", "ingestion_timestamp", "source_file")
)

df_silver = (
    df_products
    .join(df_translation, "product_category_name", "left")

    # Fix column name typos
    .withColumnRenamed("product_name_lenght", "product_name_length")
    .withColumnRenamed("product_description_lenght", "product_description_length")

    # Fill missing category names
    .withColumn(
        "product_category_name_english",
        coalesce(col("product_category_name_english"), lit("unknown"))
    )

    .dropDuplicates(["product_id"])
)

df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.products")

print(f"Rows: {df_silver.count():,}")

In [0]:
display(df_silver)

In [0]:
#  7. SELLERS
print("\n--- silver.sellers ---")

df = spark.table("bronze.sellers")

df_silver = (
    df
    .withColumn("seller_city", initcap(trim(col("seller_city"))))
    .withColumn("seller_state", upper(trim(col("seller_state"))))
    .withColumn(
        "seller_zip_code_prefix",
        col("seller_zip_code_prefix").cast("string")
    )
    .dropDuplicates(["seller_id"])
)

df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.sellers")
print(f"Rows: {df_silver.count():,}")

In [0]:
# 8. GEOLOCATION 
print("\n--- silver.geolocation ---")

df = spark.table("bronze.geolocation")

df_silver = (
    df
    # Reduce huge dataset (~1M rows) → group by ZIP
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        avg("geolocation_lat").alias("latitude"),
        avg("geolocation_lng").alias("longitude"),
        first("geolocation_city").alias("city"),
        first("geolocation_state").alias("state")
    )
    .withColumn(
        "geolocation_zip_code_prefix",
        col("geolocation_zip_code_prefix").cast("string")
    )
)

df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.geolocation")
print(f"Rows: {df_silver.count():,} (reduced from ~1M)")


In [0]:
# 9. CATEGORY TRANSLATION 
print("\n--- silver.category_translation ---")

df = spark.table("bronze.category_translation")

df.write.format("delta").mode("overwrite").saveAsTable("silver.category_translation")
print(f"Rows: {df.count():,}")


print("\n" + "="*60)
print("Silver layer transformations COMPLETE!")
print("="*60)